### What it does
Script to handle words with "-", for example "risk-taker". The aim is to modify it into "risk taker", "risktaker". 
It searchs "-" in words ignoring case. Then it split the words by "-" and joint the words in one taking into account lowcase letter.
It creates new table with the the same columns as original table. Original table with new one are merged and sorted by lemma. 

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

# pandas library to work with tables/dataframes
import pandas as pd


# Regular expression library
import re
import ast

In [34]:
#file name
file_name = "rules_31JAN_English"

In [35]:
#read csv file to the pandas table
df_rules = pd.read_csv(file_name+".csv")

In [36]:
#get info about the table
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1154 entries, 0 to 1153
Data columns (total 22 columns):
Lemma                     1154 non-null object
Word_Type                 1154 non-null object
Priority                  649 non-null object
Category                  1154 non-null object
Primary_subcategory       1154 non-null object
Secondary_subcategory     298 non-null object
Alt_Field                 369 non-null object
Alt_Sg_Replacement        564 non-null object
Alt_Sg_people_first       127 non-null object
Alt_Sg_reframed           212 non-null object
Alt_Pl_collective_noun    32 non-null object
Alt_Pl                    98 non-null object
Alt_Pl_people_first       6 non-null object
Alt_Pl_reframed           3 non-null object
3rd_party_alternatives    2 non-null object
3rd_party_source          159 non-null object
Sample_Sentences          3 non-null object
False_Positives           14 non-null object
Notes                     244 non-null object
ToClarify                 

In [32]:
#first 5 row of the table
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives
0,attention-seeking,a,1,gendered,female_stereotype,NaN,NaN,outgoing | at ease in social situations | greg...,NaN,causing disruption | tending to cause disruption,NaN,NaN,NaN,NaN,NaN
1,beautiful,a,NaN,gendered,female_stereotype,NaN,NaN,attractive | good-looking,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,bossy,a,NaN,gendered,female_stereotype,NaN,NaN,in charge | supervising | responsible for resu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,heroine,s,1,gendered,female_stereotype,NaN,NaN,person of the hour | person making sacrifices ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,histrionic,s,23-12-2021,gendered,female_stereotype,NaN,NaN,NaN,person with HPD | person with histrionic perso...,lively | dramatic | vivacious | enthusiastic |...,NaN,NaN,NaN,NaN,NaN


### Remove unnecessary columns

In [77]:
#create a subset of dataframe from the 'Lemma' till the last column with alternatives '3rd_party_alternatives'
df_rules=df_rules.loc[:, 'Lemma':'3rd_party_alternatives']

In [76]:
#check new table
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1154 entries, 0 to 1153
Data columns (total 15 columns):
Lemma                     1154 non-null object
Word_Type                 1154 non-null object
Priority                  649 non-null object
Category                  1154 non-null object
Primary_subcategory       1154 non-null object
Secondary_subcategory     298 non-null object
Alt_Field                 369 non-null object
Alt_Sg_Replacement        564 non-null object
Alt_Sg_people_first       127 non-null object
Alt_Sg_reframed           212 non-null object
Alt_Pl_collective_noun    32 non-null object
Alt_Pl                    98 non-null object
Alt_Pl_people_first       6 non-null object
Alt_Pl_reframed           3 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 135.4+ KB


In [18]:
#check fisrts 5 rows of the table
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives
0,attention-seeking,a,1,gendered,female_stereotype,NaN,NaN,outgoing | at ease in social situations | greg...,NaN,causing disruption | tending to cause disruption,NaN,NaN,NaN,NaN,NaN
1,beautiful,a,NaN,gendered,female_stereotype,NaN,NaN,attractive | good-looking,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,bossy,a,NaN,gendered,female_stereotype,NaN,NaN,in charge | supervising | responsible for resu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,heroine,s,1,gendered,female_stereotype,NaN,NaN,person of the hour | person making sacrifices ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,histrionic,s,23-12-2021,gendered,female_stereotype,NaN,NaN,NaN,person with HPD | person with histrionic perso...,lively | dramatic | vivacious | enthusiastic |...,NaN,NaN,NaN,NaN,NaN


In [ ]:
#check the number of columns
len(list(df_rules.columns))

### Splitting and merging words by hyphen

In [29]:
list_word = []

list_word_type = []
list_priority = []
list_category = []
list_primary_subcategory = []
list_secondary_subcategory = []
list_alt_field= []
list_alt_sg_replacement = []
list_alt_sg_people_first = []
list_alt_sg_reframed = []
list_alt_pl_collective_noun = []
list_alt_pl = []
list_alt_pl_people_first = []
list_alt_pl_reframed = []
list_3rd_party_alternatives = []


for index, row in df_rules.iterrows():
    word = row["Lemma"]
    span = re.search("-", word)
    #only merge and split words if they are two, for example risk-taker (two words: risk and taker)
    if type(span) == re.Match and len(word.rsplit("-"))==2:
        #merge words by "-"
        list_word.append(word.rsplit("-")[0]+word.rsplit("-")[1].lower())
        list_word_type.append(row["Word_Type"])
        list_priority.append(row["Priority"])
        list_category.append(row["Category"])
        list_primary_subcategory.append(row["Primary_subcategory"])
        list_secondary_subcategory.append(row["Secondary_subcategory"])
        list_alt_field.append(row["Alt_Field"])
        list_alt_sg_replacement.append(row["Alt_Sg_Replacement"])
        list_alt_sg_people_first.append(row["Alt_Sg_people_first"])
        list_alt_sg_reframed.append(row["Alt_Sg_reframed"])
        list_alt_pl_collective_noun.append(row["Alt_Pl_collective_noun"])
        list_alt_pl.append(row["Alt_Pl"])
        list_alt_pl_people_first.append(row["Alt_Pl_people_first"])
        list_alt_pl_reframed.append(row['Alt_Pl_reframed']
        list_3rd_party_alternatives.append(row["3rd_party_alternatives"])
        
        
        
        #split words by "-"  
        list_word.append(word.rsplit("-")[0]+" "+word.rsplit("-")[1])
        list_word_type.append(row["Word_Type"])
        list_priority.append(row["Priority"])
        list_category.append(row["Category"])
        list_primary_subcategory.append(row["Primary_subcategory"])
        list_secondary_subcategory.append(row["Secondary_subcategory"])
        list_alt_field.append(row["Alt_Field"])
        list_alt_sg_replacement.append(row["Alt_Sg_Replacement"])
        list_alt_sg_people_first.append(row["Alt_Sg_people_first"])
        list_alt_sg_reframed.append(row["Alt_Sg_reframed"])
        list_alt_pl_collective_noun.append(row["Alt_Pl_collective_noun"])
        list_alt_pl.append(row["Alt_Pl"])
        list_alt_pl_people_first.append(row["Alt_Pl_people_first"])
        list_alt_pl_reframed.append(row['Alt_Pl_reframed'])
        list_3rd_party_alternatives.append(row["3rd_party_alternatives"])
        

### Created new dataframe from lists

In [41]:
#NEW  from the bunch of created lists create a dataframe
data = pd.DataFrame(list(zip(list_word,list_word_type,list_priority, list_category, list_primary_subcategory, list_secondary_subcategory, 
list_alt_field, list_alt_sg_replacement, list_alt_sg_people_first, list_alt_sg_reframed, list_alt_pl_collective_noun,
list_alt_pl, list_alt_pl_people_first, list_alt_pl_reframed, list_3rd_party_alternatives)), columns=list(df_rules.columns))

In [48]:
# check number of columns in the new dataframe
len(data.columns)

15

In [49]:
#list of names of the columns
data_columns = list(data.columns)

### Test name of the columns

In [45]:
for name1, name2 in zip(df_rules_columns, data_columns):
    if name1!=name2:
        print(name1, name2)
    else:
        print("correct")

correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct
correct


In [40]:
#check first 5 rows of the new dataframe
data.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives
0,attentionseeking,a,1,gendered,female_stereotype,NaN,NaN,outgoing | at ease in social situations | greg...,NaN,causing disruption | tending to cause disruption,NaN,NaN,NaN,NaN,NaN
1,attention seeking,a,1,gendered,female_stereotype,NaN,NaN,outgoing | at ease in social situations | greg...,NaN,causing disruption | tending to cause disruption,NaN,NaN,NaN,NaN,NaN
2,everyman,s,NaN,gendered,function,NaN,NaN,everybody | all,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,every man,s,NaN,gendered,function,NaN,NaN,everybody | all,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,gogetter,s,22-01-2022,gendered,function,exaggerating,NaN,bundle of energy | ball of fire,person who continues despite difficulties/oppo...,NaN,NaN,motivated people | energetic and focused colle...,NaN,NaN,NaN


In [46]:
#check the last 5 rows of the new dataframe
data.tail()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives
231,double blind review,a+s,1,unconscious_bias,visual_impairment,cultural,NaN,anonymous review | identity-hidden review | do...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
232,eyeopener,s,1,unconscious_bias,visual_impairment,cultural,NaN,utterly shocking | real surprise,NaN,"startling (situation, revelation, ...)",NaN,NaN,NaN,NaN,NaN
233,eye opener,s,1,unconscious_bias,visual_impairment,cultural,NaN,utterly shocking | real surprise,NaN,"startling (situation, revelation, ...)",NaN,NaN,NaN,NaN,NaN
234,microaggressions,s,26-01-22,inclusive,d_and_i,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
235,micro aggressions,s,26-01-22,inclusive,d_and_i,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### New created dataframe

In [50]:
# check info about new dataframe
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236 entries, 0 to 235
Data columns (total 15 columns):
Lemma                     236 non-null object
Word_Type                 236 non-null object
Priority                  106 non-null object
Category                  236 non-null object
Primary_subcategory       236 non-null object
Secondary_subcategory     58 non-null object
Alt_Field                 90 non-null object
Alt_Sg_Replacement        104 non-null object
Alt_Sg_people_first       24 non-null object
Alt_Sg_reframed           64 non-null object
Alt_Pl_collective_noun    4 non-null object
Alt_Pl                    12 non-null object
Alt_Pl_people_first       2 non-null object
Alt_Pl_reframed           0 non-null float64
3rd_party_alternatives    0 non-null float64
dtypes: float64(2), object(13)
memory usage: 27.8+ KB


In [51]:
#original dataframe
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1154 entries, 0 to 1153
Data columns (total 15 columns):
Lemma                     1154 non-null object
Word_Type                 1154 non-null object
Priority                  649 non-null object
Category                  1154 non-null object
Primary_subcategory       1154 non-null object
Secondary_subcategory     298 non-null object
Alt_Field                 369 non-null object
Alt_Sg_Replacement        564 non-null object
Alt_Sg_people_first       127 non-null object
Alt_Sg_reframed           212 non-null object
Alt_Pl_collective_noun    32 non-null object
Alt_Pl                    98 non-null object
Alt_Pl_people_first       6 non-null object
Alt_Pl_reframed           3 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 135.4+ KB


### Merge two dataframes (original and new one)

In [56]:
#merge two dataframes
df_fin = df_rules.append(data, ignore_index = True)

In [59]:
#check info about merged dataframe
df_fin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1390 entries, 0 to 1389
Data columns (total 15 columns):
Lemma                     1390 non-null object
Word_Type                 1390 non-null object
Priority                  755 non-null object
Category                  1390 non-null object
Primary_subcategory       1390 non-null object
Secondary_subcategory     356 non-null object
Alt_Field                 459 non-null object
Alt_Sg_Replacement        668 non-null object
Alt_Sg_people_first       151 non-null object
Alt_Sg_reframed           276 non-null object
Alt_Pl_collective_noun    36 non-null object
Alt_Pl                    110 non-null object
Alt_Pl_people_first       8 non-null object
Alt_Pl_reframed           3 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 163.0+ KB


### Sorted rows in dataframe by lemmas (in alphabetic order)

In [62]:
df_sorted = df_fin.sort_values(by=['Lemma'], inplace=False, ascending=True)

### Save the sorted dataframe to csv

In [63]:
#save to csv file after preprocessing
df_sorted.to_csv(file_name+"_h"+".csv", index=False)

In [64]:
#test read of dataframe
df_test = pd.read_csv(file_name+"_h"+".csv")

In [65]:
#check info
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1390 entries, 0 to 1389
Data columns (total 15 columns):
Lemma                     1390 non-null object
Word_Type                 1390 non-null object
Priority                  755 non-null object
Category                  1390 non-null object
Primary_subcategory       1390 non-null object
Secondary_subcategory     356 non-null object
Alt_Field                 459 non-null object
Alt_Sg_Replacement        668 non-null object
Alt_Sg_people_first       151 non-null object
Alt_Sg_reframed           276 non-null object
Alt_Pl_collective_noun    36 non-null object
Alt_Pl                    110 non-null object
Alt_Pl_people_first       8 non-null object
Alt_Pl_reframed           3 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 163.0+ KB


In [66]:
#check first 5 rows 
df_test.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives
0,ADD,s,1,unconscious_bias,medical_condition,NaN,NaN,attention deficit hyperactivity disorder | ADHD,person with ADHD,fidgety | restless | easily bored,NaN,NaN,NaN,NaN,NaN
1,ADHD,s,1,unconscious_bias,medical_condition,NaN,NaN,attention deficit hyperactivity disorder,person with ADHD | person with attention defic...,fidgety | restless | easily bored,NaN,NaN,NaN,NaN,NaN
2,ASD,s,1,unconscious_bias,medical_condition,NaN,NaN,neurodiversity | autism spectrum disorder,NaN,neurodevelopmental disorder characterized by d...,NaN,NaN,NaN,NaN,NaN
3,African queen,s,NaN,openly_discriminating,homophobia,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Are you blind?,i,1,unconscious_bias,visual_impairment,cultural,NaN,Don't you see?,NaN,NaN,NaN,NaN,NaN,NaN,NaN
